In [1]:
import numpy as np
from scipy.optimize import linprog

# ============================================================
# ЗАВДАННЯ 1. Математична постановка задачі
# ============================================================

# x1 — кількість виробів А на тиждень
# x2 — кількість виробів Б на тиждень
#
# Максимізувати:
# 45*x1 + 65*x2
#
# Обмеження:
# 3*x1 + 4*x2 <= 280       # сировина, кг
# 2*x1 + 6*x2 <= 210       # час, год
# 2*x1 + 2*x2 <= 230       # електроенергія, кВт·год
# x1 >= 0
# x2 >= 0


# ============================================================
# ЗАВДАННЯ 2. Розв'язання через scipy.optimize.linprog
# ============================================================

c = [-45, -65]

A_ub = [
    [3, 4],
    [2, 6],
    [2, 2]
]

b_ub = [
    280,
    210,
    230
]

bounds = [
    (0, None),
    (0, None)
]

res = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_ub,
    bounds=bounds,
    method="highs"
)

print("ЗАВДАННЯ 2")
print("Статус:", res.status)
print("Повідомлення:", res.message)

print("\nОптимальний розв'язок:")
print("x1 =", res.x[0])
print("x2 =", res.x[1])

print("\nres.fun =", res.fun)
print("Максимальний прибуток =", -res.fun, "грн")


# ============================================================
# ЗАВДАННЯ 3. Інтерпретація оптимального розв'язку
# ============================================================

x1 = res.x[0]
x2 = res.x[1]
profit = -res.fun

print("\nЗАВДАННЯ 3")

print(f"Виріб А: {x1:.2f} од./тиждень")
print(f"Виріб Б: {x2:.2f} од./тиждень")
print(f"Максимальний прибуток: {profit:.2f} грн/тиждень")

print("\nСтруктура випуску:")
print(f"А : Б = {x1:.2f} : {x2:.2f}")

print("""
Якщо кількість виробів виходить дробовою, це означає,
що стандартна задача лінійного програмування допускає
неперервні значення змінних. У реальному виробництві кількість
виробів може вимагати цілих значень, тому просте округлення
не завжди дає оптимальний цілочисловий розв'язок і може навіть
порушити ресурсні обмеження. Для гарантовано найкращого
цілочислового рішення потрібна задача цілочислового
програмування.
""")


# ============================================================
# ЗАВДАННЯ 4. Активні та неактивні обмеження
# ============================================================

print("\nЗАВДАННЯ 4")

slack = res.slack

resources = [
    "Сировина",
    "Робочий час",
    "Електроенергія"
]

limits = np.array([280, 210, 230])

usage = np.array([
    3 * x1 + 4 * x2,
    2 * x1 + 6 * x2,
    2 * x1 + 2 * x2
])

for i in range(3):
    print(f"\n{resources[i]}:")
    print(f"Використано: {usage[i]:.2f}")
    print(f"Ліміт: {limits[i]:.2f}")
    print(f"Запас (slack): {slack[i]:.2f}")

    if np.isclose(slack[i], 0):
        print("Статус: АКТИВНЕ обмеження")
    else:
        print("Статус: НЕАКТИВНЕ обмеження")


# ============================================================
# ЗАВДАННЯ 5. What-if аналіз
# ============================================================

print("\nЗАВДАННЯ 5")

# ------------------------------------------------------------
# 5.1. Збільшення активного ресурсу на 15%
# ------------------------------------------------------------

active_index = np.argmin(slack)

active_names = [
    "сировини",
    "робочого часу",
    "електроенергії"
]

new_b_15 = b_ub.copy()
new_b_15[active_index] *= 1.15

res_active = linprog(
    c,
    A_ub=A_ub,
    b_ub=new_b_15,
    bounds=bounds,
    method="highs"
)

print("\n5.1. Збільшення активного ресурсу на 15%")

print(
    f"Збільшуємо ліміт {active_names[active_index]} "
    f"на 15%."
)

print("\nБуло:")
print(f"x1 = {x1:.2f}")
print(f"x2 = {x2:.2f}")
print(f"Прибуток = {profit:.2f} грн")

print("\nСтало:")
print(f"x1 = {res_active.x[0]:.2f}")
print(f"x2 = {res_active.x[1]:.2f}")
print(f"Прибуток = {-res_active.fun:.2f} грн")

print(
    f"\nЗміна прибутку = "
    f"{-res_active.fun - profit:.2f} грн"
)


# ------------------------------------------------------------
# 5.2. Збільшення неактивного ресурсу на 50%
# ------------------------------------------------------------

inactive_index = np.argmax(slack)

new_b_50 = b_ub.copy()
new_b_50[inactive_index] *= 1.50

res_inactive = linprog(
    c,
    A_ub=A_ub,
    b_ub=new_b_50,
    bounds=bounds,
    method="highs"
)

print("\n5.2. Збільшення неактивного ресурсу на 50%")

print(
    f"Збільшуємо ліміт {active_names[inactive_index]} "
    f"на 50%."
)

print("\nБуло:")
print(f"x1 = {x1:.2f}")
print(f"x2 = {x2:.2f}")
print(f"Прибуток = {profit:.2f} грн")

print("\nСтало:")
print(f"x1 = {res_inactive.x[0]:.2f}")
print(f"x2 = {res_inactive.x[1]:.2f}")
print(f"Прибуток = {-res_inactive.fun:.2f} грн")

print(
    f"\nЗміна прибутку = "
    f"{-res_inactive.fun - profit:.2f} грн"
)


# ============================================================
# ПІДСУМКОВА ТАБЛИЦЯ
# ============================================================

print("\n" + "=" * 60)
print("ПІДСУМОК")
print("=" * 60)

print(f"Оптимальний випуск А: {x1:.2f} од.")
print(f"Оптимальний випуск Б: {x2:.2f} од.")
print(f"Максимальний прибуток: {profit:.2f} грн")

print("\nЗапаси ресурсів:")
for i in range(3):
    print(f"{resources[i]}: {slack[i]:.2f}")

print("\nПісля збільшення активного ресурсу на 15%:")
print(f"А: {res_active.x[0]:.2f}")
print(f"Б: {res_active.x[1]:.2f}")
print(f"Прибуток: {-res_active.fun:.2f} грн")

print("\nПісля збільшення неактивного ресурсу на 50%:")
print(f"А: {res_inactive.x[0]:.2f}")
print(f"Б: {res_inactive.x[1]:.2f}")
print(f"Прибуток: {-res_inactive.fun:.2f} грн")


ЗАВДАННЯ 2
Статус: 0
Повідомлення: Optimization terminated successfully. (HiGHS Status 7: Optimal)

Оптимальний розв'язок:
x1 = 84.0
x2 = 7.0

res.fun = -4235.0
Максимальний прибуток = 4235.0 грн

ЗАВДАННЯ 3
Виріб А: 84.00 од./тиждень
Виріб Б: 7.00 од./тиждень
Максимальний прибуток: 4235.00 грн/тиждень

Структура випуску:
А : Б = 84.00 : 7.00

Якщо кількість виробів виходить дробовою, це означає,
що стандартна задача лінійного програмування допускає
неперервні значення змінних. У реальному виробництві кількість
виробів може вимагати цілих значень, тому просте округлення
не завжди дає оптимальний цілочисловий розв'язок і може навіть
порушити ресурсні обмеження. Для гарантовано найкращого
цілочислового рішення потрібна задача цілочислового
програмування.


ЗАВДАННЯ 4

Сировина:
Використано: 280.00
Ліміт: 280.00
Запас (slack): 0.00
Статус: АКТИВНЕ обмеження

Робочий час:
Використано: 210.00
Ліміт: 210.00
Запас (slack): 0.00
Статус: АКТИВНЕ обмеження

Електроенергія:
Використано: 182.00
Лі

Контрольні питання
1. Чому linprog мінімізує, якщо в задачі потрібно максимізувати прибуток?
scipy.optimize.linprog призначена для мінімізації цільової функції. Тому для задачі максимізації прибутку потрібно змінити знак коефіцієнтів цільової функції:

max
⁡
(
45
x
1
+
65
x
2
)

перетворюємо на

min
⁡
(
−
45
x
1
−
65
x
2
)
.

Після отримання res.fun справжній максимальний прибуток дорівнює -res.fun.

2. Чому обмеження >= потрібно домножити на -1?
linprog приймає ресурсні обмеження у формі:

A
u
b
x
≤
b
u
b
.

Тому обмеження виду

3
x
1
+
4
x
2
≥
280

потрібно помножити на -1:

−
3
x
1
−
4
x
2
≤
−
280.

У нашому варіанті всі ресурсні обмеження вже мають форму <=, тому змінювати їх знак не потрібно.

3. Що таке активне та неактивне обмеження?
Активне обмеження — це обмеження, яке в оптимальній точці використовується повністю. Його slack = 0. Такий ресурс є вузьким місцем виробництва.

Неактивне обмеження має запас, тобто slack > 0. Збільшення такого ресурсу саме по собі не повинно покращувати оптимальний прибуток, оскільки він не обмежує поточний оптимальний план.

4. Чи узгоджується What-if аналіз із поняттям вузького місця?
Так. Якщо збільшити ліміт активного ресурсу, оптимальний прибуток може зрости, оскільки усувається частина обмеження виробництва.

Якщо збільшити ліміт неактивного ресурсу, оптимальний прибуток не повинен змінитися, оскільки цей ресурс і до збільшення мав запас і не був вузьким місцем.


